In [7]:
# %%
# A/B de recomendações em marketplace — simulação + testes
# Rodar célula por célula em um notebook. Não precisa de internet.

import numpy as np
import pandas as pd
from scipy import stats
from dataclasses import dataclass

rng = np.random.default_rng(42)  # troque a seed p/ novos cenários

# -----------------------------
# 1) GERADOR DE DADOS (parâmetros ocultos aqui dentro)
# -----------------------------

@dataclass
class SimConfig:
    n_users: int = 80_000
    assign_rate_B: float = 0.5   # split A/B
    exposure_rate: float = 0.65  # % que de fato vê o módulo de recoms
    days: int = 7                # janela do experimento
    impressions_per_day_mu: float = 6.0  # média de impressões por dia (Poisson)


def simulate_ab(config: SimConfig, rng=np.random.default_rng()):
    n = config.n_users

    # Segmentos e covariáveis (pré-experimentais)
    is_new = rng.binomial(1, 0.35, size=n)                         # usuário novo (35%)
    device = rng.choice(["mobile", "desktop"], p=[0.7, 0.3], size=n)
    region = rng.choice(["SE", "S", "NE", "CO", "N"], p=[0.5,0.2,0.15,0.1,0.05], size=n)

    # Histórico pré-experimento (para CUPED)
    # "Clicks" e "Impressões" anteriores a t0
    pre_impr = rng.poisson(30, size=n)
    # baseline CTR histórico heterogêneo
    base_ctr_user = stats.logistic.cdf(
        -1.2
        + 0.7*is_new
        + 0.3*(device=="mobile")
        + rng.normal(0, 0.6, size=n)
    )
    pre_clicks = rng.binomial(pre_impr, np.clip(base_ctr_user, 1e-6, 1-1e-6))
    pre_ctr = np.divide(pre_clicks, np.maximum(pre_impr, 1), where=(pre_impr>0))

    # Atribuição
    arm = rng.binomial(1, config.assign_rate_B, size=n)  # 0=A, 1=B
    exposed = rng.binomial(1, config.exposure_rate, size=n)  # viu recoms?

    # Impressões na janela
    impr = rng.poisson(config.impressions_per_day_mu * config.days, size=n) * exposed

    # --- Verdade "oculta" (não olhe! rs) ---
    # CTR baseline por usuário (sem tratamento)
    ctr_A = stats.logistic.cdf(
        -1.0
        + 0.8*is_new
        + 0.25*(device=="mobile")
        + 0.15*(region=="SE")
        + rng.normal(0, 0.5, size=n)
    )

    # Efeito do ranker B (uplift em probabilidade de clique)
    # maior em usuários novos e mobile
    uplift_ctr = (
        0.06*is_new + 0.025*(device=="mobile") + rng.normal(0, 0.005, size=n)
    )
    ctr = np.clip(ctr_A + arm*uplift_ctr, 1e-6, 1-1e-6)

    # Clicks
    clicks = rng.binomial(impr, ctr)

    # Conversão condicional ao clique (CVR)
    cvr_A = stats.logistic.cdf(
        -2.0 + 0.5*(device=="desktop") + 0.2*(region=="SE") + rng.normal(0,0.5,size=n)
    )
    uplift_cvr = (0.01*(device=="mobile") + 0.015*is_new + rng.normal(0,0.003,size=n))
    cvr = np.clip(cvr_A + arm*uplift_cvr, 1e-6, 1-1e-6)

    conv = rng.binomial(clicks, cvr)

    # Valor de pedido (GMV) por conversão (lognormal heterogênea)
    order_value = np.exp(
        rng.normal(
            3.0 + 0.10*(device=="desktop") + 0.05*(region=="SE") + 0.15*is_new,
            0.45,
            size=n
        )
    )
    gmv = conv * order_value

    df = pd.DataFrame({
        "user_id": np.arange(n),
        "arm": np.where(arm==1, "B", "A"),
        "exposed": exposed,
        "impr": impr,
        "clicks": clicks,
        "conv": conv,
        "gmv": gmv,
        "device": device,
        "region": region,
        "is_new": is_new.astype(int),
        "pre_impr": pre_impr,
        "pre_clicks": pre_clicks,
        "pre_ctr": pre_ctr,
    })
    return df

config = SimConfig()
df = simulate_ab(config, rng)
df.head()


,user_id,arm,exposed,impr,clicks,conv,gmv,device,region,is_new,pre_impr,pre_clicks,pre_ctr
0,0,B,0,0,0,0,0.000000,mobile,S,1,28,13,0.464286
1,1,B,1,42,9,1,28.076158,mobile,S,0,38,11,0.289474
2,2,A,1,40,19,2,58.114379,mobile,SE,1,19,3,0.157895
3,3,A,1,39,27,2,92.447962,mobile,SE,1,26,8,0.307692
4,4,B,1,40,15,1,13.833638,mobile,SE,0,35,17,0.485714


In [8]:
exp = df.loc[df["exposed"].eq(1)].copy()

exp["ctr"] = exp["clicks"].div(exp["impr"]).where(exp["impr"] > 0, other=0.0)
exp["cvr"] = exp["conv"].div(exp["clicks"]).where(exp["clicks"] > 0, other=0.0)
exp["gmv_per_user"] = exp["gmv"]


In [9]:
exp

,user_id,arm,exposed,impr,clicks,conv,gmv,device,region,is_new,pre_impr,pre_clicks,pre_ctr,ctr,cvr,gmv_per_user
1,1,B,1,42,9,1,28.076158,mobile,S,0,38,11,0.289474,0.214286,0.111111,28.076158
2,2,A,1,40,19,2,58.114379,mobile,SE,1,19,3,0.157895,0.475000,0.105263,58.114379
3,3,A,1,39,27,2,92.447962,mobile,SE,1,26,8,0.307692,0.692308,0.074074,92.447962
4,4,B,1,40,15,1,13.833638,mobile,SE,0,35,17,0.485714,0.375000,0.066667,13.833638
5,5,B,1,47,10,4,45.858453,desktop,S,1,31,19,0.612903,0.212766,0.400000,45.858453
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,79995,A,1,40,26,10,148.462520,mobile,SE,1,43,17,0.395349,0.650000,0.384615,148.462520
79996,79996,A,1,47,17,3,91.103200,mobile,S,0,28,7,0.250000,0.361702,0.176471,91.103200
79997,79997,A,1,42,18,0,0.000000,desktop,CO,0,26,4,0.153846,0.428571,0.000000,0.000000
79998,79998,B,1,42,25,3,64.248993,mobile,NE,1,24,15,0.625000,0.595238,0.120000,64.248993


In [10]:
# %%
# 3) TESTES FREQUENCISTAS
# 3.1) Proporções (CTR e CVR): teste de diferença de proporções (bicaudal)

def two_prop_test(success_a, total_a, success_b, total_b):
    # pooled
    p_pool = (success_a + success_b) / (total_a + total_b)
    se = np.sqrt(p_pool*(1-p_pool)*(1/total_a + 1/total_b))
    z = ((success_b/total_b) - (success_a/total_a)) / se
    p = 2*(1 - stats.norm.cdf(abs(z)))
    return z, p

A = exp.query("arm=='A'")
B = exp.query("arm=='B'")

z_ctr, p_ctr = two_prop_test(A["clicks"].sum(), A["impr"].sum(),
                             B["clicks"].sum(), B["impr"].sum())
z_cvr, p_cvr = two_prop_test(A["conv"].sum(), A["clicks"].sum(),
                             B["conv"].sum(), B["clicks"].sum())

print(f"CTR diff test: z={z_ctr:.3f}, p={p_ctr:.2e}")
print(f"CVR diff test: z={z_cvr:.3f}, p={p_cvr:.2e}")

# 3.2) GMV por usuário: Welch t-test (variâncias desiguais)
t_gmv, p_gmv = stats.ttest_ind(B["gmv_per_user"], A["gmv_per_user"], equal_var=False)
print(f"GMV/user Welch t-test: t={t_gmv:.3f}, p={p_gmv:.2e}")


CTR diff test: z=57.565, p=0.00e+00
CVR diff test: z=18.868, p=0.00e+00
GMV/user Welch t-test: t=18.826, p=8.54e-79


In [11]:
# %%
# 5) INTERVALOS DE CONFIANÇA por bootstrap (GMV por usuário)
def bootstrap_mean_ci(x, n_boot=3000, alpha=0.05, rng=rng):
    bs = rng.choice(x, size=(n_boot, x.size), replace=True).mean(axis=1)
    lo, hi = np.quantile(bs, [alpha/2, 1-alpha/2])
    return x.mean(), (lo, hi)

mean_A, ci_A = bootstrap_mean_ci(A["gmv_per_user"].values)
mean_B, ci_B = bootstrap_mean_ci(B["gmv_per_user"].values)
uplift = mean_B - mean_A
print(f"GMV/user A: {mean_A:.2f}  CI95% [{ci_A[0]:.2f}, {ci_A[1]:.2f}]")
print(f"GMV/user B: {mean_B:.2f}  CI95% [{ci_B[0]:.2f}, {ci_B[1]:.2f}]")
print(f"Uplift (B-A): {uplift:.2f}")


GMV/user A: 67.22  CI95% [66.34, 68.13]
GMV/user B: 80.39  CI95% [79.38, 81.46]
Uplift (B-A): 13.16
